# Coordinate Reference Systems

**Series:** Geospatial Python Basics (notebook 2 of 4)
**Time:** about 25 minutes

**In this notebook you will:**
- Check the coordinate reference system (CRS) of a layer
- Explain why distances and areas need a projected CRS
- Reproject with `to_crs()` and measure in meters
- Tell `set_crs()` and `to_crs()` apart

**You should already know:** GeoDataFrames (notebook 04a)

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

Coordinates are just numbers. The **coordinate reference system** (CRS) says what they mean: degrees of
latitude and longitude, or meters east and north of some origin. Every layer has one, and layers must share
one before you combine or measure them. The Data Management module's Coordinate Reference Systems episode
explains the theory; here we practice it.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

counties = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/indiana_counties.geojson")
counties.crs

**EPSG:4326** is latitude and longitude on the WGS 84 datum, the usual CRS for web data and GPS. Its units
are **degrees**.

## Why degrees are a problem

A degree of longitude is about 111 km at the equator but only about 85 km in Indiana, and it shrinks to zero
at the poles. So areas and distances calculated in degrees are meaningless. GeoPandas even warns you:

In [ ]:
counties[counties["name"] == "Tippecanoe"].area      # square degrees: not a useful unit

## Reproject with to_crs()

`to_crs()` converts the coordinates to another CRS. Two good choices for Indiana:

| EPSG code | Name | Units | Good for |
|---|---|---|---|
| 4326 | WGS 84 (latitude and longitude) | degrees | Storing and sharing data |
| 5070 | USA Contiguous Albers Equal Area | meters | Areas and maps of the whole U.S. or a state |
| 26916 | NAD83 / UTM zone 16N | meters | Distances and local analysis in Indiana (zone 16 covers 84 to 90 degrees west, all of Indiana) |

In [ ]:
counties_albers = counties.to_crs(epsg=5070)
counties_albers["area_km2"] = counties_albers.area / 1_000_000      # square meters to square kilometers
counties_albers["census_km2"] = counties_albers["land_sq_mi"] * 2.59
counties_albers[["name", "area_km2", "census_km2"]].head().round(0)

The calculated areas are close to the Census Bureau's land areas. They are not identical: the Census
figures leave out water, and the boundary file is simplified.

The map looks different too, because each CRS draws the shapes differently:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 6))
counties.plot(ax=axes[0], edgecolor="white")
axes[0].set_title("EPSG:4326 (degrees)")
counties_albers.plot(ax=axes[1], edgecolor="white")
axes[1].set_title("EPSG:5070 (meters)")
plt.show()

## Measuring a distance

Make two points in latitude and longitude, reproject them, and measure. `points_from_xy()` takes
**longitude first** (x), then latitude (y).

In [ ]:
places = gpd.GeoDataFrame(
    {"name": ["Purdue Memorial Union", "Indiana Statehouse"]},
    geometry=gpd.points_from_xy([-86.9107, -86.1626], [40.4247, 39.7684]),
    crs="EPSG:4326",
)
places_utm = places.to_crs(epsg=26916)
meters = places_utm.geometry.iloc[0].distance(places_utm.geometry.iloc[1])
print(f"{meters / 1000:.1f} km in a straight line")

That is the straight-line distance; the drive is longer, as the network analysis lessons show.

## set_crs() is not to_crs()

- `to_crs()` **converts** coordinates from the CRS the layer has to a new one. The numbers change.
- `set_crs()` only **labels** a layer whose CRS is missing. The numbers do not change.

Use `set_crs()` only when a file has no CRS and you know what it should be (from its metadata). Labeling a
layer with the wrong CRS is a classic mistake: the numbers stay the same but are now read in the wrong
units, so the layer lands in the wrong place. Below, Indiana's degrees are read as meters, and the whole
state shrinks to a speck near the equator, in the Pacific Ocean west of South America.

In [ ]:
print(counties.total_bounds.round(2))          # west, south, east, north in degrees

wrong = counties.set_crs(epsg=26916, allow_override=True)   # wrong label: degrees read as meters
print(wrong.to_crs(epsg=4326).total_bounds.round(2))        # now the "counties" sit near the equator

## Checklist before combining layers

1. Check each layer's `.crs`.
2. If they differ, `to_crs()` one to match the other.
3. Before measuring distances or areas, `to_crs()` to a projected CRS in meters.

## Try it yourself

1. Calculate the area of Tippecanoe County in square kilometers using **EPSG:26916**.
2. Compare it with the Census land area (`land_sq_mi` times 2.59).

<details>
<summary><b>Show a solution</b></summary>

```python
tip = counties[counties["name"] == "Tippecanoe"].to_crs(epsg=26916)
print(round(tip.area.iloc[0] / 1_000_000, 1), "km2 calculated")
print(round(tip["land_sq_mi"].iloc[0] * 2.59, 1), "km2 from the Census")
```

</details>

In [ ]:
# Write your answer here

## Recap

- `.crs` tells you a layer's coordinate reference system
- Degrees (EPSG:4326) are fine for storing data but not for measuring
- `to_crs()` converts coordinates; use a projected CRS in meters (5070, 26916) for areas and distances
- `set_crs()` only labels data with a missing CRS; it never moves anything

**Next:** [Joining Tables and Layers](https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/04c_Joining_Tables_and_Layers.ipynb)